# Online Retail II: data quality and sales diagnostics

**Work in progress — first public milestone.** Historical portfolio analysis using the UCI Online Retail II dataset. This notebook preserves the step-by-step investigation; SQL, repeat-purchase analysis, charts/dashboard and the final report remain planned.

## Summary
- The two sheets overlap; retaining the overlap only from the second sheet yields 1,044,848 rows.
- Candidate product transactions total 1,033,062 rows and GBP 18,981,261.52 net recorded amount.
- April 2011 net recorded product amount is 24.42% below April 2010. This is a descriptive accounting comparison, not a causal estimate.
- A large cancellation is followed 12 minutes later by a positive invoice. Cancellation amounts must not automatically be described as lost demand or physical returns.

## Context and methods
Source: [Chen, D. (2012), Online Retail II, UCI](https://doi.org/10.24432/C5CG6D), licensed CC BY 4.0. Observation window: 2009-12-01 to 2011-12-09. Unit: invoice line; currency: GBP. Customer IDs are identifiers supplied in the public dataset.

### Assumptions and limitations
Exact within-sheet duplicates remain in the main population; the earlier sensitivity check is on raw recorded amounts, not every final metric. Known postage, financial adjustments, manual entries, samples, tests and vouchers are excluded from candidate product amounts. Unclassified codes are candidates, not independently verified products. Positive quantity, positive price and a non-C invoice define candidate positive sales. C invoices with negative quantity and positive price define candidate cancellations. Cancellation dates determine the month; original sales are not systematically linked. No costs are available, so net recorded amount is not profit. Orders count distinct positive invoices, not fulfilled orders. Customer comparisons exclude missing IDs and are not churn estimates. Stock-code case is preserved. Display names are presentation labels only. December 2011 is excluded from monthly comparisons; other absent dates do not prove missing data or closures.

## Run
Install `requirements.txt`, download the original workbook, put `online_retail_II.xlsx` beside this notebook, select the Python kernel and **Restart & Run All**. Run from this directory. Raw data is not included in the repository. Saved outputs are derived from the cited public data.


## Data loading

In [1]:
import pandas as pd

file_path = "online_retail_II.xlsx"
excel = pd.ExcelFile(file_path)

excel.sheet_names

Out[1]: ['Year 2009-2010', 'Year 2010-2011']


In [2]:
df1 = pd.read_excel(excel,sheet_name="Year 2009-2010")
df2 = pd.read_excel(excel,sheet_name="Year 2010-2011")

print("First table: ", df1.shape)
print("Second table: ", df2.shape)

print("Attributes of first table: ", df1.columns.tolist())
print("Attributes of second table: ", df2.columns.tolist())

df1.head()



First table:  (525461, 8)
Second table:  (541910, 8)
Attributes of first table:  ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Attributes of second table:  ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
Out[2]: 
  Invoice StockCode  ... Customer ID         Country
0  489434     85048  ...     13085.0  United Kingdom
1  489434    79323P  ...     13085.0  United Kingdom
2  489434    79323W  ...     13085.0  United Kingdom
3  489434     22041  ...     13085.0  United Kingdom
4  489434     21232  ...     13085.0  United Kingdom

[5 rows x 8 columns]


## Schema and missingness

In [3]:
df1.info()
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 32.1+ MB
<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      541910 non-null  object        
 1   StockCode    541910 non-null  object        
 2   Descrip

In [4]:
print("# of missing values in first table: ")
print(df1.isna().sum())

print("# of missing values in second table: ")
print(df2.isna().sum())

# of missing values in first table: 
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64
# of missing values in second table: 
Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64


In [5]:
print("ratio of missing value in first table (%): ")
print((df1.isna().mean()*100).round(2))
print("ratio of missing value in second table (%): ")
print((df2.isna().sum() / len(df2) *100).round(2))


ratio of missing value in first table (%): 
Invoice         0.00
StockCode       0.00
Description     0.56
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    20.54
Country         0.00
dtype: float64
ratio of missing value in second table (%): 
Invoice         0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    24.93
Country         0.00
dtype: float64


In [6]:
print(df1["InvoiceDate"].min(), df1["InvoiceDate"].max())
print(df2["InvoiceDate"].min(), df2["InvoiceDate"].max())

2009-12-01 07:45:00 2010-12-09 20:01:00
2010-12-01 08:26:00 2011-12-09 12:50:00


## Duplicate records

In [7]:
print("Duplicated rows in first table: ",df1.duplicated().sum())
print("Duplicated rows in second table: ",df2.duplicated().sum())

Duplicated rows in first table:  6865
Duplicated rows in second table:  5268


In [8]:
df1[df1.duplicated(keep=False)].sort_values(
    ["Invoice", "StockCode"]
).head(20)

Out[8]: 
    Invoice StockCode  ... Customer ID         Country
379  489517     21491  ...     16329.0  United Kingdom
391  489517     21491  ...     16329.0  United Kingdom
365  489517     21821  ...     16329.0  United Kingdom
386  489517     21821  ...     16329.0  United Kingdom
363  489517     21912  ...     16329.0  United Kingdom
371  489517     21912  ...     16329.0  United Kingdom
394  489517     21912  ...     16329.0  United Kingdom
362  489517     21913  ...     16329.0  United Kingdom
385  489517     21913  ...     16329.0  United Kingdom
368  489517     22130  ...     16329.0  United Kingdom
383  489517     22130  ...     16329.0  United Kingdom
367  489517     22319  ...     16329.0  United Kingdom
384  489517     22319  ...     16329.0  United Kingdom
388  489517    84951A  ...     16329.0  United Kingdom
390  489517    84951A  ...     16329.0  United Kingdom
634  489529     22028  ...     17984.0  United Kingdom
657  489529     22028  ...     17984.0  United Kingdom
6

In [9]:
commonRows = df1.drop_duplicates().merge(
    df2.drop_duplicates(),
    # similar to on in SQL, at this time we are ON everything should be matched each other from column list
    on=df1.columns.tolist(),
    #inner join
    how="inner"
)

print("# of Distinct records shared by both tables: ", len(commonRows))
commonRows.head()

# of Distinct records shared by both tables:  22202
Out[9]: 
  Invoice StockCode  ... Customer ID         Country
0  536365    85123A  ...     17850.0  United Kingdom
1  536365     71053  ...     17850.0  United Kingdom
2  536365    84406B  ...     17850.0  United Kingdom
3  536365    84029G  ...     17850.0  United Kingdom
4  536365    84029E  ...     17850.0  United Kingdom

[5 rows x 8 columns]


In [10]:
print(commonRows["InvoiceDate"].min())
print(commonRows["InvoiceDate"].max())

2010-12-01 08:26:00
2010-12-09 20:01:00


## Quantity and price checks

In [11]:
for name, df in [("Sheet 1", df1), ("Sheet 2", df2)]:
    print(name)
    print("Quantity < 0:", (df["Quantity"] < 0).sum())
    print("Quantity = 0:", (df["Quantity"] == 0).sum())
    print("Price < 0:", (df["Price"] < 0).sum())
    print("Price = 0:", (df["Price"] == 0).sum())
    

Sheet 1
Quantity < 0: 12326
Quantity = 0: 0
Price < 0: 3
Price = 0: 3687
Sheet 2
Quantity < 0: 10624
Quantity = 0: 0
Price < 0: 2
Price = 0: 2515


In [12]:
#check negative-price records

df1[df1["Price" ]< 0]

Out[12]: 
        Invoice StockCode  ... Customer ID         Country
179403  A506401         B  ...         NaN  United Kingdom
276274  A516228         B  ...         NaN  United Kingdom
403472  A528059         B  ...         NaN  United Kingdom

[3 rows x 8 columns]


In [13]:
#check negative-price records
df2[df2["Price"]<0]

Out[13]: 
        Invoice StockCode  ... Customer ID         Country
299983  A563186         B  ...         NaN  United Kingdom
299984  A563187         B  ...         NaN  United Kingdom

[2 rows x 8 columns]


In [14]:
#check 0-price records
df1[df1["Price"] == 0].head(10)

Out[14]: 
     Invoice StockCode   Description  ...  Price Customer ID         Country
263   489464     21733  85123a mixed  ...    0.0         NaN  United Kingdom
283   489463     71477         short  ...    0.0         NaN  United Kingdom
284   489467    85123A   21733 mixed  ...    0.0         NaN  United Kingdom
470   489521     21646           NaN  ...    0.0         NaN  United Kingdom
3114  489655     20683           NaN  ...    0.0         NaN  United Kingdom
3161  489659     21350           NaN  ...    0.0         NaN  United Kingdom
3162  489660     35956          lost  ...    0.0         NaN  United Kingdom
3168  489663    35605A       damages  ...    0.0         NaN  United Kingdom
3731  489781     84292           NaN  ...    0.0         NaN  United Kingdom
4296  489806     18010           NaN  ...    0.0         NaN  United Kingdom

[10 rows x 8 columns]


In [15]:
#check 0-price records
df2[df2["Price"] == 0].head(10)

Out[15]: 
     Invoice StockCode Description  ...  Price Customer ID         Country
622   536414     22139         NaN  ...    0.0         NaN  United Kingdom
1510  536545     21134         NaN  ...    0.0         NaN  United Kingdom
1985  536547     37509         NaN  ...    0.0         NaN  United Kingdom
1986  536546     22145         NaN  ...    0.0         NaN  United Kingdom
2022  536552     20950         NaN  ...    0.0         NaN  United Kingdom
2023  536549    85226A         NaN  ...    0.0         NaN  United Kingdom
2024  536550     85044         NaN  ...    0.0         NaN  United Kingdom
2025  536553     37461         NaN  ...    0.0         NaN  United Kingdom
2026  536554     84670         NaN  ...    0.0         NaN  United Kingdom
2406  536589     21777         NaN  ...    0.0         NaN  United Kingdom

[10 rows x 8 columns]


In [16]:
for name,df in [("Sheet 1", df1),("Sheet 2", df2)]:
    records = df[(df["Price"] == 0) & (df["Customer ID"].notna())]
    print(name, " # of records with customerid but 0-price: ", len(records))
    display(records.head())

Sheet 1  # of records with customerid but 0-price:  31
Sheet 2  # of records with customerid but 0-price:  40


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom


In [17]:
for name,df in [("Sheet 1",df1),("Sheet 2",df2)]:
    cancelled = df["Invoice"].astype(str).str.upper().str.startswith("C")
    negative = df["Quantity"] < 0

    print(name)
    print("Negative quantity + C invoice: ",(negative&cancelled).sum())
    print("Negative quantity + non-C invoice: ", (negative&~cancelled).sum())

Sheet 1
Negative quantity + C invoice:  10205
Negative quantity + non-C invoice:  2121
Sheet 2
Negative quantity + C invoice:  9288
Negative quantity + non-C invoice:  1336


In [18]:
for name, df in [("Sheet 1", df1), ("Sheet 2", df2)]:
    cancelled = df["Invoice"].astype(str).str.upper().str.startswith("C")
    records = df[(df["Quantity"] < 0) & ~cancelled]

    print(name)
    display(records.head(10))

Sheet 1
Sheet 2


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,2010-12-03 15:32:00,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,2010-12-03 15:33:00,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,2010-12-03 15:34:00,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,2010-12-03 15:35:00,0.0,NaN,United Kingdom


In [19]:
for name, df in [("Sheet 1", df1), ("Sheet 2", df2)]:
    cancelled = df["Invoice"].astype(str).str.upper().str.startswith("C")
    records = df[(df["Quantity"] < 0) & ~cancelled]

    print(name)
    print("Total rows:", len(records))
    print("Zero-price rows:", (records["Price"] == 0).sum())
    print("Missing customer ID:", records["Customer ID"].isna().sum())

Sheet 1
Total rows: 2121
Zero-price rows: 2121
Missing customer ID: 2121
Sheet 2
Total rows: 1336
Zero-price rows: 1336
Missing customer ID: 1336


The two sheets contain 2,121 and 1,336 rows with negative quantities but no cancellation flag. All these rows have a price of zero and no customer ID. Some descriptions mention lost, damaged, or missing items, suggesting they may be inventory adjustments. We should not automatically count them as customer returns.

In [20]:
for name, df in [("Sheet 1", df1), ("Sheet 2", df2)]:
    cancelled = df["Invoice"].astype(str).str.upper().str.startswith("C")

    print(name)
    print("C invoice rows:", cancelled.sum())
    print("C invoice rows with quantity >= 0:",(cancelled & (df["Quantity"] >= 0)).sum())

Sheet 1
C invoice rows: 10206
C invoice rows with quantity >= 0: 1
Sheet 2
C invoice rows: 9288
C invoice rows with quantity >= 0: 0


In [21]:
cancelled = df1["Invoice"].astype(str).str.upper().str.startswith("C")

df1[cancelled & (df1["Quantity"] >= 0)]

Out[21]: 
       Invoice StockCode Description  ...   Price Customer ID         Country
76799  C496350         M      Manual  ...  373.57         NaN  United Kingdom

[1 rows x 8 columns]


Invoice C496350 is marked as cancelled but has a positive quantity of 1 and a price of £373.57. Its description is “Manual”, and the customer ID is missing. The reason is unclear, so we will flag it for separate review and exclude it from regular sales and cancellation totals.

## Verify overlap before combining sheets

In [22]:

start = max(df1["InvoiceDate"].min(), df2["InvoiceDate"].min())
end = min(df1["InvoiceDate"].max(), df2["InvoiceDate"].max())


overlap1 = df1[df1["InvoiceDate"].between(start, end)].drop_duplicates()
overlap2 = df2[df2["InvoiceDate"].between(start, end)].drop_duplicates()

# 
comparison = overlap1.merge(
    overlap2,
    on=df1.columns.tolist(),
    how="outer",
    indicator=True
)

print(comparison["_merge"].value_counts())

_merge
both          22202
left_only         0
right_only        0
Name: count, dtype: int64


In [23]:
counts1 = (
    df1[df1["InvoiceDate"].between(start, end)]
    .value_counts(dropna=False)
)

counts2 = (
    df2[df2["InvoiceDate"].between(start, end)]
    .value_counts(dropna=False)
)

counts1, counts2 = counts1.align(counts2, join="left")
print("Records with different counts:", (counts1 - counts2).ne(0).sum())

Records with different counts: 0


In [24]:
counts1, counts2 = counts1.align(counts2, join="left")
print("Records with different counts:", (counts1 - counts2).ne(0).sum())

Records with different counts: 0


In [25]:
combined = pd.concat(
    [
        df1[df1["InvoiceDate"] < start].assign(source_sheet="Year 2009-2010"),
        df2.assign(source_sheet="Year 2010-2011")
    ],
    ignore_index=True
)

print("Combined shape:", combined.shape)

Combined shape: (1044848, 9)


In [26]:
combined.head()

Out[26]: 
  Invoice StockCode  ...         Country    source_sheet
0  489434     85048  ...  United Kingdom  Year 2009-2010
1  489434    79323P  ...  United Kingdom  Year 2009-2010
2  489434    79323W  ...  United Kingdom  Year 2009-2010
3  489434     22041  ...  United Kingdom  Year 2009-2010
4  489434     21232  ...  United Kingdom  Year 2009-2010

[5 rows x 9 columns]


In [27]:
combined["source_sheet"].value_counts()

Out[27]: 
source_sheet
Year 2010-2011    541910
Year 2009-2010    502938
Name: count, dtype: int64


The overlapping period contains the same records with the same occurrence counts in both sheets. We kept this period from the second sheet and retained only earlier records from the first sheet. The combined dataset contains 1,044,848 rows. Duplicates within each sheet have not been removed.

## Duplicate sensitivity

In [28]:
original_columns = df1.columns.tolist()

duplicate_count = combined.duplicated(
    subset=original_columns
).sum()

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_count / len(combined) * 100, 2), "%")

Duplicate rows: 11812
Duplicate percentage: 1.13 %


In [29]:
# 
deduplicated = combined.drop_duplicates(subset=original_columns)

amount_before = (combined["Quantity"] * combined["Price"]).sum()
amount_after = (deduplicated["Quantity"] * deduplicated["Price"]).sum()

print("Amount before:", round(amount_before, 2))
print("Amount after:", round(amount_after, 2))
print("Difference:", round(amount_before - amount_after, 2))

Amount before: 18909762.12
Amount after: 18855533.7
Difference: 54228.42


Removing 11,812 exact duplicate rows reduces the total recorded amount by £54,228.42 (about 0.29%). Without a unique line-item ID, we cannot confirm whether these are duplicate entries or valid repeated items. We will retain them in the main dataset and use the deduplicated version to check how much the results change.

## Identifier types and transaction flags

In [30]:
combined[["Invoice", "StockCode", "Customer ID"]].dtypes

Out[30]: 
Invoice         object
StockCode       object
Customer ID    float64
dtype: object


In [31]:
combined["Invoice"] = combined["Invoice"].astype("string")
combined["StockCode"] = combined["StockCode"].astype("string")

combined["Customer ID"] = (
    combined["Customer ID"].astype("Int64").astype("string")
)

combined[["Invoice", "StockCode", "Customer ID"]].head()

Out[31]: 
  Invoice StockCode Customer ID
0  489434     85048       13085
1  489434    79323P       13085
2  489434    79323W       13085
3  489434     22041       13085
4  489434     21232       13085


In [32]:
combined["is_cancelled"] = (
    combined["Invoice"].str.upper().str.startswith("C")
)

combined["line_amount"] = combined["Quantity"] * combined["Price"]

combined[
    ["Invoice", "Quantity", "Price", "is_cancelled", "line_amount"]
].head()

Out[32]: 
  Invoice  Quantity  Price  is_cancelled  line_amount
0  489434        12   6.95         False         83.4
1  489434        12   6.75         False         81.0
2  489434        12   6.75         False         81.0
3  489434        48   2.10         False        100.8
4  489434        24   1.25         False         30.0


In [33]:
combined["is_positive_sale"] = (
    (combined["Quantity"] > 0)
    & (combined["Price"] > 0)
    & ~combined["is_cancelled"]
)

combined["is_positive_sale"].value_counts()

Out[33]: 
is_positive_sale
True     1019654
False      25194
Name: count, dtype: Int64


## Investigate special stock codes

In [34]:
special = combined[
    combined["StockCode"].str.fullmatch(r"[A-Za-z]+", na=False)
]

special.groupby(
    ["StockCode", "Description"], dropna=False
).size().reset_index(name="row_count")

Out[34]: 
    StockCode                          Description  row_count
0      ADJUST  Adjustment by Peter on 24/05/2010 1          3
1      ADJUST  Adjustment by john on 26/01/2010 16         38
2      ADJUST  Adjustment by john on 26/01/2010 17         26
3   AMAZONFEE                           AMAZON FEE         36
4           B                      Adjust bad debt          6
5        CRUK                      CRUK Commission         16
6           D                             Discount        173
7    DCGSLBOY                                  NaN          1
8   DCGSLGIRL                                  NaN          1
9    DCGSSBOY                       BOYS PARTY BAG         21
10   DCGSSBOY                               update          1
11   DCGSSBOY                                  NaN          1
12  DCGSSGIRL                      GIRLS PARTY BAG         23
13  DCGSSGIRL                               update          1
14  DCGSSGIRL                                  NaN          

In [35]:
combined.loc[
    combined["StockCode"] == "B",
    ["Invoice", "Description", "Quantity", "Price", "line_amount"]
]

Out[35]: 
        Invoice      Description  Quantity     Price  line_amount
179403  A506401  Adjust bad debt         1 -53594.36    -53594.36
276274  A516228  Adjust bad debt         1 -44031.79    -44031.79
403472  A528059  Adjust bad debt         1 -38925.87    -38925.87
802920  A563185  Adjust bad debt         1  11062.06     11062.06
802921  A563186  Adjust bad debt         1 -11062.06    -11062.06
802922  A563187  Adjust bad debt         1 -11062.06    -11062.06


In [36]:
combined["is_bad_debt"] = combined["StockCode"].eq("B")

combined.loc[
    combined["is_bad_debt"],
    ["Invoice", "line_amount", "is_positive_sale", "is_bad_debt"]
]

Out[36]: 
        Invoice  line_amount  is_positive_sale  is_bad_debt
179403  A506401    -53594.36             False         True
276274  A516228    -44031.79             False         True
403472  A528059    -38925.87             False         True
802920  A563185     11062.06              True         True
802921  A563186    -11062.06             False         True
802922  A563187    -11062.06             False         True


In [37]:
combined["is_postage"] = combined["StockCode"].isin(["POST", "DOT"])

combined.loc[
    combined["is_postage"],
    ["StockCode", "Description", "line_amount", "is_postage"]
].head()

Out[37]: 
    StockCode Description  line_amount  is_postage
89       POST     POSTAGE        54.00        True
126      POST     POSTAGE       141.00        True
173      POST     POSTAGE       130.00        True
625      POST     POSTAGE       108.00        True
927      POST     POSTAGE        -9.58        True


In [38]:
combined["is_manual"] = (
    combined["StockCode"].str.upper().eq("M")
)

combined.loc[
    combined["is_manual"],
    ["Invoice", "Description", "Quantity", "Price", "line_amount"]
].head()

Out[38]: 
      Invoice Description  Quantity    Price  line_amount
2697   489609      Manual         1     4.00         4.00
3053  C489651      Manual        -1     5.10        -5.10
5897  C489859      Manual        -1    69.57       -69.57
9259  C490126      Manual        -1     5.95        -5.95
9307  C490129      Manual        -1  1998.49     -1998.49


In [39]:
special_types = {
    "B": "bad_debt",
    "POST": "postage",
    "DOT": "postage",
    "M": "manual",
    "D": "discount",
    "ADJUST": "adjustment",
    "AMAZONFEE": "platform_fee",
    "CRUK": "commission",
    "S": "samples"
}

combined["record_type"] = (
    combined["StockCode"].str.upper()
    .map(special_types)
    .fillna("unclassified")
)

combined["record_type"].value_counts()

Out[39]: 
record_type
unclassified    1039534
postage            3511
manual             1403
discount            173
samples             102
adjustment           67
platform_fee         36
commission           16
bad_debt              6
Name: count, dtype: int64


In [40]:
standard_code = combined["StockCode"].str.fullmatch(
    r"\d{5}[A-Za-z]*", na=False
)

other_codes = combined[
    combined["record_type"].eq("unclassified") & ~standard_code
]

other_codes.groupby(
    ["StockCode", "Description"], dropna=False
).size().reset_index(name="row_count")

Out[40]: 
       StockCode                          Description  row_count
0        47503J      SET/3 FLORAL GARDEN TOOLS IN BAG          1
1        ADJUST2  Adjustment by Peter on Jun 25 2010           3
2   BANK CHARGES                         Bank Charges          6
3   BANK CHARGES                         Bank Charges         94
4             C2                             CARRIAGE        274
..           ...                                  ...        ...
74  gift_0001_70   Dotcomgiftshop Gift Voucher £70.00          1
75  gift_0001_70                                  NaN          2
76  gift_0001_80   Dotcomgiftshop Gift Voucher £80.00          2
77  gift_0001_80                                  NaN          2
78  gift_0001_90                                  NaN          2

[79 rows x 3 columns]


In [41]:
summary = (
    other_codes.groupby(
        ["StockCode", "Description"], dropna=False
    ).size().reset_index(name="row_count")
)

with pd.option_context("display.max_rows", None):
    display(summary)

,StockCode,Description,row_count
0,47503J,SET/3 FLORAL GARDEN TOOLS IN BAG,1
1,ADJUST2,Adjustment by Peter on Jun 25 2010,3
2,BANK CHARGES,Bank Charges,6
3,BANK CHARGES,Bank Charges,94
4,C2,CARRIAGE,274
5,C2,NaN,3
6,C3,NaN,1
7,DCGS0003,BOXED GLASS ASHTRAY,13
8,DCGS0003,ebay,1
9,DCGS0004,HAYNES CAMPER SHOULDER BAG,4


In [42]:
codes = combined["StockCode"].drop_duplicates()

has_spaces = codes.ne(codes.str.strip())

print(codes[has_spaces].map(repr).to_string(index=False))

'47503J '


In [43]:
combined["StockCode"] = combined["StockCode"].str.strip()

In [44]:
combined.loc[
    combined["StockCode"].eq("BANK CHARGES"),
    "Description"
].drop_duplicates().map(repr)

Out[44]: 
18410     ' Bank Charges'
114180     'Bank Charges'
Name: Description, dtype: str


In [45]:
combined["Description"] = combined["Description"].str.strip()

In [46]:
special_types.update({
    "ADJUST2": "adjustment",
    "BANK CHARGES": "bank_fee",
    "C2": "postage",
    "TEST001": "test",
    "TEST002": "test"
})

combined["record_type"] = (
    combined["StockCode"].str.upper()
    .map(special_types)
    .fillna("unclassified")
)

# 
gift_voucher = combined["StockCode"].str.upper().str.startswith(
    "GIFT_0001_", na=False
)
combined.loc[gift_voucher, "record_type"] = "gift_voucher"

# 
combined["is_postage"] = combined["record_type"].eq("postage")

combined["record_type"].value_counts()

Out[46]: 
record_type
unclassified    1039037
postage            3788
manual             1403
discount            173
samples             102
bank_fee            100
gift_voucher        100
adjustment           70
platform_fee         36
test                 17
commission           16
bad_debt              6
Name: count, dtype: int64


In [47]:
combined.loc[
    combined["StockCode"].isin(["C3", "GIFT"]),
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "line_amount"]
]

Out[47]: 
       Invoice StockCode Description  Quantity  Price Customer ID  line_amount
83304   497107      GIFT         NaN        -9    0.0        <NA>         -0.0
302041  518785        C3         NaN     -4999    0.0        <NA>         -0.0


In [48]:
combined.loc[
    combined["StockCode"].isin(["C3", "GIFT"]),
    "record_type"
] = "needs_review"

In [49]:
combined.loc[
    combined["StockCode"].isin(["C3", "GIFT"]),
    ["StockCode", "record_type"]
]

Out[49]: 
       StockCode   record_type
83304       GIFT  needs_review
302041        C3  needs_review


## Candidate product sales and large transactions

In [50]:
combined["is_product_sale_candidate"] = (
    combined["is_positive_sale"]
    & combined["record_type"].eq("unclassified")
)

combined["is_product_sale_candidate"].value_counts()

Out[50]: 
is_product_sale_candidate
True     1015088
False      29760
Name: count, dtype: Int64


In [51]:
combined.loc[
    combined["is_product_sale_candidate"],
    ["Invoice", "StockCode", "Description", "Quantity", "Price", "line_amount", "Customer ID"]
].nlargest(10, "line_amount")

Out[51]: 
        Invoice StockCode  ... line_amount  Customer ID
1043359  581483     23843  ...   168469.60        16446
564557   541431     23166  ...    77183.60        12346
725609   556444     22502  ...    38970.00        15098
432176   530715     84347  ...    15818.40        15838
228042   511465    15044A  ...     8925.00        18008
851263   567423     23243  ...     7144.72        17450
555649   540815     21108  ...     6539.40        15749
663484   550461     21108  ...     6539.40        15749
461644   533027     22086  ...     5803.25         <NA>
379875   525968     84347  ...     5179.20        15838

[10 rows x 7 columns]


In [52]:
matched = (
    (
        combined["Customer ID"].eq("16446")
        & combined["StockCode"].eq("23843")
    )
    | (
        combined["Customer ID"].eq("12346")
        & combined["StockCode"].eq("23166")
    )
)

combined.loc[
    matched,
    ["Invoice", "InvoiceDate", "Customer ID", "StockCode",
     "Quantity", "Price", "line_amount", "is_cancelled"]
].sort_values(["Customer ID", "InvoiceDate"])

Out[52]: 
         Invoice         InvoiceDate  ... line_amount is_cancelled
564557    541431 2011-01-18 10:01:00  ...     77183.6        False
564562   C541433 2011-01-18 10:17:00  ...    -77183.6         True
1043359   581483 2011-12-09 09:15:00  ...    168469.6        False
1043360  C581484 2011-12-09 09:27:00  ...   -168469.6         True

[4 rows x 8 columns]


The two largest candidate product sales each have a cancellation recorded shortly afterward on the same day, with the same customer, product, and unit price, but the opposite quantity. Each pair sums to zero. We will retain these records rather than remove them solely because of their size, and report both positive sales amounts and the impact of cancellations.

## Candidate cancellations and amounts

In [53]:
combined["is_product_cancellation_candidate"] = (
    combined["is_cancelled"]
    & (combined["Quantity"] < 0)
    & (combined["Price"] > 0)
    & combined["record_type"].eq("unclassified")
)

combined["is_product_cancellation_candidate"].value_counts()

Out[53]: 
is_product_cancellation_candidate
False    1026874
True       17974
Name: count, dtype: Int64


In [54]:
positive_amount = combined.loc[
    combined["is_product_sale_candidate"], "line_amount"
].sum()

cancellation_amount = combined.loc[
    combined["is_product_cancellation_candidate"], "line_amount"
].sum()

net_amount = positive_amount + cancellation_amount

print("Positive product amount:", round(positive_amount, 2))
print("Cancellation amount:", round(cancellation_amount, 2))
print("Net product amount:", round(net_amount, 2))

Positive product amount: 19700954.46
Cancellation amount: -719692.94
Net product amount: 18981261.52


## Customer coverage and descriptions

In [55]:
sales = combined.loc[combined["is_product_sale_candidate"]]

print("Sales rows:", len(sales))
print("Missing customer ID:", sales["Customer ID"].isna().sum())
print(
    "Missing customer ID (%):",
    round(sales["Customer ID"].isna().mean() * 100, 2)
)

Sales rows: 1015088
Missing customer ID: 226882
Missing customer ID (%): 22.35


In [56]:
anonymous_amount = sales.loc[
    sales["Customer ID"].isna(), "line_amount"
].sum()

print("Amount without customer ID:", round(anonymous_amount, 2))
print(
    "Share of positive product amount (%):",
    round(anonymous_amount / positive_amount * 100, 2)
)

Amount without customer ID: 2576013.46
Share of positive product amount (%): 13.08


In [57]:
print("Missing descriptions:", sales["Description"].isna().sum())

print(
    "Blank descriptions:",
    sales["Description"].eq("").sum()
)

Missing descriptions: 0
Blank descriptions: 0


In [58]:
description_counts = sales.groupby("StockCode")["Description"].nunique()

print("Codes with multiple descriptions:", (description_counts > 1).sum())

description_counts[description_counts > 1].sort_values(
    ascending=False
).head(10)

Codes with multiple descriptions: 622
Out[58]: 
StockCode
22344     4
22345     4
22346     4
20685     4
22384     4
23196     4
23236     4
21524     3
17107D    3
22785     3
Name: Description, dtype: int64


In [59]:
sales.loc[
    sales["StockCode"].eq("22344"),
    "Description"
].value_counts()

Out[59]: 
Description
PARTY PIZZA DISH PINK POLKADOT      89
PARTY PIZZA DISH PINK RETROSPOT     24
PARTY PIZZA DISH PINK WHITE SPOT    19
PARTY PIZZA DISH PINK+WHITE SPOT     2
Name: count, dtype: int64


## Display names

In [60]:
product_names = (
    sales.groupby("StockCode")["Description"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index(name="product_name")
)

product_names.head()

Out[60]: 
  StockCode                product_name
0     10002  INFLATABLE POLITICAL GLOBE
1    10002R       ROBOT PENCIL SHARPNER
2     10080    GROOVY CACTUS INFLATABLE
3     10109        BENDY COLOUR PENCILS
4     10120                DOGGY RUBBER


In [61]:
combined = combined.drop(columns=["product_name"], errors="ignore").merge(
    product_names,
    on="StockCode",
    how="left",
    validate="many_to_one"
)

combined[
    ["StockCode", "Description", "product_name"]
].head()

Out[61]: 
  StockCode  ...                         product_name
0     85048  ...  15CM CHRISTMAS GLASS BALL 20 LIGHTS
1    79323P  ...                   PINK CHERRY LIGHTS
2    79323W  ...                  WHITE CHERRY LIGHTS
3     22041  ...          RECORD FRAME 7" SINGLE SIZE
4     21232  ...       STRAWBERRY CERAMIC TRINKET BOX

[5 rows x 3 columns]


In [62]:
missing_names = combined[
    combined["is_product_cancellation_candidate"]
    & combined["product_name"].isna()
]

print("Cancellation rows without product name:", len(missing_names))

missing_names[
    ["Invoice", "StockCode", "Description", "line_amount"]
].head(10)

Cancellation rows without product name: 21
Out[62]: 
       Invoice StockCode                          Description  line_amount
933    C489541     21254        SET OF KITCHEN WALL  STICKERS       -11.80
14055  C490513    79340W           WHITE ORCHID FLOWER LIGHTS        -7.95
16203  C490739     22003       VINTAGE BLUE VACUUM FLASK 0.5L       -67.50
18391  C490942     21701  SET 6 MINI SUSHI SET FRIDGE MAGNETS       -20.28
19526  C490997     21053         ZINC HEART HANGER WITH HOOKS        -4.25
19557  C491005    79340W           WHITE ORCHID FLOWER LIGHTS        -7.95
24210  C491402    35631B    BLUE GREEN CHRISTMAS HANGING BALL        -0.19
25303  C491521    79340P          PURPLE ORCHID FLOWER LIGHTS        -7.95
25516  C491606    35001C               HAND OPEN SHAPE CHROME       -10.17
25680  C491626     21766               VINTAGE WOOD ORGANISER       -33.90


In [63]:
mask = (
    combined["is_product_cancellation_candidate"]
    & combined["product_name"].isna()
)

combined.loc[mask, "product_name"] = combined.loc[mask, "Description"]

print(
    "Still missing:",
    combined.loc[mask, "product_name"].isna().sum()
)

Still missing: 0


## Analysis population and monthly coverage

In [64]:
product_transactions = combined.loc[
    combined["is_product_sale_candidate"]
    | combined["is_product_cancellation_candidate"]
].copy()

print("Rows:", len(product_transactions))
print("Net amount:", round(product_transactions["line_amount"].sum(), 2))

Rows: 1033062
Net amount: 18981261.52


In [65]:
monthly_coverage = (
    product_transactions
    .groupby(product_transactions["InvoiceDate"].dt.to_period("M"))
    .agg(
        first_date=("InvoiceDate", "min"),
        last_date=("InvoiceDate", "max"),
        row_count=("InvoiceDate", "size")
    )
)

monthly_coverage

Out[65]: 
                     first_date           last_date  row_count
InvoiceDate                                                   
2009-12     2009-12-01 07:45:00 2009-12-23 16:58:00      44808
2010-01     2010-01-04 09:43:00 2010-01-31 16:02:00      31119
2010-02     2010-02-01 08:13:00 2010-02-28 16:16:00      28698
2010-03     2010-03-01 08:38:00 2010-03-31 17:31:00      40901
2010-04     2010-04-01 07:49:00 2010-04-30 17:33:00      33647
2010-05     2010-05-02 10:22:00 2010-05-30 15:58:00      34593
2010-06     2010-06-01 09:02:00 2010-06-30 17:38:00      39449
2010-07     2010-07-01 08:39:00 2010-07-30 17:08:00      33028
2010-08     2010-08-01 10:04:00 2010-08-31 17:44:00      32855
2010-09     2010-09-01 09:25:00 2010-09-30 20:11:00      41647
2010-10     2010-10-01 08:19:00 2010-10-31 16:26:00      58505
2010-11     2010-11-01 08:56:00 2010-11-30 19:35:00      77347
2010-12     2010-12-01 08:26:00 2010-12-23 17:41:00      42013
2011-01     2011-01-04 10:00:00 2011-01-31 16

In [66]:
monthly_transactions = product_transactions.loc[
    product_transactions["InvoiceDate"] < "2011-12-01"
].copy()

print("Rows for monthly analysis:", len(monthly_transactions))

Rows for monthly analysis: 1007683


## Monthly trend and year-over-year comparison

In [67]:
monthly_net = (
    monthly_transactions
    .groupby(monthly_transactions["InvoiceDate"].dt.to_period("M"))
    ["line_amount"]
    .sum()
    .round(2)
)

monthly_net

Out[67]: 
InvoiceDate
2009-12     781541.69
2010-01     606385.18
2010-02     527160.30
2010-03     754676.65
2010-04     638968.56
2010-05     608351.25
2010-06     672935.41
2010-07     617764.26
2010-08     663642.07
2010-09     844100.37
2010-10    1072467.65
2010-11    1400559.24
2010-12     760323.91
2011-01     580406.89
2011-02     500510.33
2011-03     680582.31
2011-04     482901.21
2011-05     732244.18
2011-06     724991.03
2011-07     677955.20
2011-08     702583.98
2011-09    1013430.68
2011-10    1062677.66
2011-11    1432734.99
Freq: M, Name: line_amount, dtype: float64


In [68]:
monthly_yoy = monthly_net.to_frame(name="net_amount")

monthly_yoy["yoy_pct"] = (
    monthly_yoy["net_amount"].pct_change(12) * 100
).round(2)

monthly_yoy

Out[68]: 
             net_amount  yoy_pct
InvoiceDate                     
2009-12       781541.69      NaN
2010-01       606385.18      NaN
2010-02       527160.30      NaN
2010-03       754676.65      NaN
2010-04       638968.56      NaN
2010-05       608351.25      NaN
2010-06       672935.41      NaN
2010-07       617764.26      NaN
2010-08       663642.07      NaN
2010-09       844100.37      NaN
2010-10      1072467.65      NaN
2010-11      1400559.24      NaN
2010-12       760323.91    -2.71
2011-01       580406.89    -4.28
2011-02       500510.33    -5.06
2011-03       680582.31    -9.82
2011-04       482901.21   -24.42
2011-05       732244.18    20.37
2011-06       724991.03     7.74
2011-07       677955.20     9.74
2011-08       702583.98     5.87
2011-09      1013430.68    20.06
2011-10      1062677.66    -0.91
2011-11      1432734.99     2.30


## April diagnostic: positive amounts and cancellations

In [69]:
april = monthly_transactions[
    monthly_transactions["InvoiceDate"].dt.month.eq(4)
]

april_comparison = april.groupby(
    [april["InvoiceDate"].dt.year, "is_cancelled"]
)["line_amount"].sum().unstack(fill_value=0).round(2)

april_comparison

Out[69]: 
is_cancelled      False     True 
InvoiceDate                      
2010          648993.82 -10025.26
2011          516218.12 -33316.91


In [70]:
april_sales = april[~april["is_cancelled"]]

april_orders = april_sales.groupby(
    april_sales["InvoiceDate"].dt.year
).agg(
    positive_amount=("line_amount", "sum"),
    order_count=("Invoice", "nunique")
)

april_orders["amount_per_order"] = (
    april_orders["positive_amount"] / april_orders["order_count"]
)

april_orders.round(2)

Out[70]: 
             positive_amount  order_count  amount_per_order
InvoiceDate                                                
2010               648993.82         1435            452.26
2011               516218.12         1235            417.99


In [71]:
active_days = april_sales.groupby(
    april_sales["InvoiceDate"].dt.year
)["InvoiceDate"].agg(lambda x: x.dt.normalize().nunique())

april_orders["active_days"] = active_days
april_orders["orders_per_active_day"] = (
    april_orders["order_count"] / april_orders["active_days"]
)

april_orders.round(2)

Out[71]: 
             positive_amount  order_count  ...  active_days  orders_per_active_day
InvoiceDate                                ...                                    
2010               648993.82         1435  ...           23                  62.39
2011               516218.12         1235  ...           21                  58.81

[2 rows x 5 columns]


In April 2011, positive product sales value fell by 20.46% year over year. The number of days with sales dropped from 23 to 21, average orders per active day fell by about 5.74%, and average order value decreased by about 7.58%. The decline therefore coincided with fewer active sales days, fewer orders per active day, and lower spending per order. The underlying business reasons still need further investigation.

## Country and customer decomposition

In [72]:
april_country = april_sales.groupby(
    ["Country", april_sales["InvoiceDate"].dt.year]
)["line_amount"].sum().unstack(fill_value=0)

april_country["change"] = april_country[2011] - april_country[2010]

april_country.sort_values("change").head(10).round(2)

Out[72]: 
InvoiceDate          2010       2011    change
Country                                       
United Kingdom  558114.09  466132.15 -91981.94
EIRE             23357.63    7270.50 -16087.13
Sweden           15868.44     649.10 -15219.34
Germany          21667.23   10994.79 -10672.44
Netherlands       6336.78    2886.56  -3450.22
France            7144.98    3899.31  -3245.67
Cyprus            1151.70       0.00  -1151.70
Austria           1388.54     584.78   -803.76
Italy             1525.85     737.83   -788.02
Denmark            748.00       0.00   -748.00


In [73]:
uk_april = april_sales[
    april_sales["Country"].eq("United Kingdom")
]

uk_comparison = uk_april.groupby(
    uk_april["InvoiceDate"].dt.year
).agg(
    positive_amount=("line_amount", "sum"),
    order_count=("Invoice", "nunique")
)

uk_comparison["amount_per_order"] = (
    uk_comparison["positive_amount"] / uk_comparison["order_count"]
)

uk_comparison.round(2)

Out[73]: 
             positive_amount  order_count  amount_per_order
InvoiceDate                                                
2010               558114.09         1328            420.27
2011               466132.15         1148            406.04


In [74]:
uk_customer_coverage = uk_april.groupby(
    [
        uk_april["InvoiceDate"].dt.year,
        uk_april["Customer ID"].notna()
    ]
)["line_amount"].sum().unstack(fill_value=0).round(2)

uk_customer_coverage

Out[74]: 
Customer ID     False      True 
InvoiceDate                     
2010         55504.24  502609.85
2011         59301.57  406830.58


In [75]:
uk_identified = uk_april[uk_april["Customer ID"].notna()]

uk_customers = uk_identified.groupby(
    uk_identified["InvoiceDate"].dt.year
).agg(
    positive_amount=("line_amount", "sum"),
    customer_count=("Customer ID", "nunique")
)

uk_customers["amount_per_customer"] = (
    uk_customers["positive_amount"] / uk_customers["customer_count"]
)

uk_customers.round(2)

Out[75]: 
             positive_amount  customer_count  amount_per_customer
InvoiceDate                                                      
2010               502609.85             873               575.73
2011               406830.58             784               518.92


In [76]:
customers_2010 = set(
    uk_identified.loc[
        uk_identified["InvoiceDate"].dt.year.eq(2010), "Customer ID"
    ]
)

customers_2011 = set(
    uk_identified.loc[
        uk_identified["InvoiceDate"].dt.year.eq(2011), "Customer ID"
    ]
)

print("Both Aprils:", len(customers_2010 & customers_2011))
print("Only April 2010:", len(customers_2010 - customers_2011))
print("Only April 2011:", len(customers_2011 - customers_2010))

Both Aprils: 261
Only April 2010: 612
Only April 2011: 523


In [77]:
shared_customers = customers_2010 & customers_2011

shared_sales = uk_identified[
    uk_identified["Customer ID"].isin(shared_customers)
]

shared_sales.groupby(
    shared_sales["InvoiceDate"].dt.year
)["line_amount"].sum().round(2)

Out[77]: 
InvoiceDate
2010    226701.67
2011    150551.70
Name: line_amount, dtype: float64


In [78]:
shared_customer_amounts = shared_sales.groupby(
    ["Customer ID", shared_sales["InvoiceDate"].dt.year]
)["line_amount"].sum().unstack(fill_value=0)

shared_customer_amounts["change"] = (
    shared_customer_amounts[2011] - shared_customer_amounts[2010]
)

shared_customer_amounts.sort_values("change").head(10).round(2)

Out[78]: 
InvoiceDate      2010     2011    change
Customer ID                             
18102        16596.55  1196.96 -15399.59
15061         9209.60    81.60  -9128.00
13694        11712.96  2948.29  -8764.67
14680         3893.00    72.00  -3821.00
17511         6631.27  2941.80  -3689.47
13881         3854.06   309.88  -3544.18
17920         3768.75   417.93  -3350.82
16684         6912.76  4018.36  -2894.40
17677         3022.45   310.00  -2712.45
16029         3739.20  1340.40  -2398.80


In [79]:
top_customers = ["18102", "15061", "13694"]

customer_trend = sales.loc[
    sales["Customer ID"].isin(top_customers)
    & sales["Country"].eq("United Kingdom")
    & sales["InvoiceDate"].dt.month.isin([3, 4, 5])
]

customer_trend.groupby(
    ["Customer ID", customer_trend["InvoiceDate"].dt.to_period("M")]
)["line_amount"].sum().unstack(fill_value=0).round(2)

Out[79]: 
InvoiceDate   2010-03   2010-04   2010-05  2011-03  2011-04   2011-05
Customer ID                                                          
13694        29300.70  11712.96   8656.57  8089.22  2948.29   6391.68
15061            0.00   9209.60   2755.20  4321.63    81.60   3648.36
18102        21215.11  16596.55  13174.25  6909.32  1196.96  18165.74


In [80]:
three_month_comparison = customer_trend.groupby(
    ["Customer ID", customer_trend["InvoiceDate"].dt.year]
)["line_amount"].sum().unstack(fill_value=0)

three_month_comparison["change"] = (
    three_month_comparison[2011] - three_month_comparison[2010]
)

three_month_comparison["change_pct"] = (
    three_month_comparison["change"]
    / three_month_comparison[2010] * 100
)

three_month_comparison.round(2)

Out[80]: 
InvoiceDate      2010      2011    change  change_pct
Customer ID                                          
13694        49670.23  17429.19 -32241.04      -64.91
15061        11964.80   8051.59  -3913.21      -32.71
18102        50985.91  26272.02 -24713.89      -48.47


The three customers contributing most to the April decline among customers who purchased in both Aprils also recorded lower positive purchase totals across March–May 2011. These totals fell by 64.91%, 32.71%, and 48.47% year over year, respectively. Purchases in the surrounding months therefore did not fully offset the April decline. We recommend that the business team review these customers’ purchasing needs, order cycles, and service experiences. The available data does not establish the underlying causes.

## Cancellation concentration and invoice investigation

In [81]:
april_cancellations = april[april["is_cancelled"]]

cancellation_by_country = april_cancellations.groupby(
    ["Country", april_cancellations["InvoiceDate"].dt.year]
)["line_amount"].sum().unstack(fill_value=0)

cancellation_by_country["change"] = (
    cancellation_by_country[2011] - cancellation_by_country[2010]
)

cancellation_by_country.sort_values("change").head(10).round(2)

Out[81]: 
InvoiceDate              2010      2011    change
Country                                          
United Kingdom       -9287.63 -30540.02 -21252.39
Japan                   -2.75  -1676.67  -1673.92
Australia                0.00   -438.20   -438.20
Switzerland            -37.00   -254.75   -217.75
France                 -40.07   -133.10    -93.03
Czech Republic           0.00    -57.51    -57.51
Finland                  0.00    -22.11    -22.11
Cyprus                 -15.80    -35.80    -20.00
Belgium                -21.25    -29.33     -8.08
United Arab Emirates   -16.50      0.00     16.50


In [82]:
uk_cancellations = april_cancellations[
    april_cancellations["Country"].eq("United Kingdom")
]

cancel_summary = uk_cancellations.groupby(
    uk_cancellations["InvoiceDate"].dt.year
).agg(
    cancellation_amount=("line_amount", "sum"),
    cancellation_invoice_count=("Invoice", "nunique")
)

cancel_summary["amount_per_cancellation"] = (
    cancel_summary["cancellation_amount"].abs()
    / cancel_summary["cancellation_invoice_count"]
)

cancel_summary.round(2)

Out[82]: 
             cancellation_amount  ...  amount_per_cancellation
InvoiceDate                       ...                         
2010                    -9287.63  ...                    38.86
2011                   -30540.02  ...                   155.03

[2 rows x 3 columns]


In [83]:
cancellations_2011 = uk_cancellations[
    uk_cancellations["InvoiceDate"].dt.year.eq(2011)
]

largest_cancellations = cancellations_2011.groupby(
    "Invoice"
)["line_amount"].sum().sort_values().head(10)

largest_cancellations

Out[83]: 
Invoice
C550456   -22998.40
C548577     -933.92
C551510     -415.00
C548972     -355.12
C550181     -330.34
C548748     -300.85
C549544     -275.00
C549549     -250.00
C550024     -175.40
C549968     -155.52
Name: line_amount, dtype: float64


In [84]:
combined.loc[
    combined["Invoice"].eq("C550456"),
    ["Invoice", "InvoiceDate", "Customer ID", "StockCode",
     "Description", "Quantity", "Price", "line_amount", "record_type"]
]

Out[84]: 
        Invoice         InvoiceDate  ... line_amount   record_type
663079  C550456 2011-04-18 13:08:00  ...     -4522.5  unclassified
663080  C550456 2011-04-18 13:08:00  ...     -3315.0  unclassified
663081  C550456 2011-04-18 13:08:00  ...     -4921.5  unclassified
663082  C550456 2011-04-18 13:08:00  ...     -3700.0  unclassified
663083  C550456 2011-04-18 13:08:00  ...     -6539.4  unclassified

[5 rows x 9 columns]


In [85]:
cancelled_codes = combined.loc[
    combined["Invoice"].eq("C550456"), "StockCode"
].unique()

customer_history = combined.loc[
    combined["Customer ID"].eq("15749")
    & combined["StockCode"].isin(cancelled_codes),
    ["Invoice", "InvoiceDate", "StockCode",
     "Quantity", "Price", "line_amount"]
].sort_values(["StockCode", "InvoiceDate"])

display(customer_history)

,Invoice,InvoiceDate,StockCode,Quantity,Price,line_amount
555649,540815,2011-01-11 12:55:00,21108,3114,2.10,6539.4
663083,C550456,2011-04-18 13:08:00,21108,-3114,2.10,-6539.4
663484,550461,2011-04-18 13:20:00,21108,3114,2.10,6539.4
555648,540815,2011-01-11 12:55:00,21175,2000,1.85,3700.0
663082,C550456,2011-04-18 13:08:00,21175,-2000,1.85,-3700.0
663483,550461,2011-04-18 13:20:00,21175,2000,1.69,3380.0
663080,C550456,2011-04-18 13:08:00,47566B,-1300,2.55,-3315.0
555710,540818,2011-01-11 12:57:00,48185,670,6.75,4522.5
663079,C550456,2011-04-18 13:08:00,48185,-670,6.75,-4522.5
663482,550461,2011-04-18 13:20:00,48185,670,6.35,4254.5


In [86]:
combined.loc[
    combined["Invoice"].eq("550461"),
    ["Invoice", "InvoiceDate", "Customer ID", "StockCode",
     "Description", "Quantity", "Price", "line_amount"]
]

Out[86]: 
       Invoice         InvoiceDate Customer ID  ... Quantity Price  line_amount
663480  550461 2011-04-18 13:20:00       15749  ...     1930  2.40       4632.0
663481  550461 2011-04-18 13:20:00       15749  ...     1300  2.10       2730.0
663482  550461 2011-04-18 13:20:00       15749  ...      670  6.35       4254.5
663483  550461 2011-04-18 13:20:00       15749  ...     2000  1.69       3380.0
663484  550461 2011-04-18 13:20:00       15749  ...     3114  2.10       6539.4

[5 rows x 8 columns]


In [87]:
combined.loc[
    combined["StockCode"].str.upper().eq("47566B")
].groupby(
    ["StockCode", "Description"], dropna=False
).size().reset_index(name="row_count")

Out[87]: 
  StockCode                          Description  row_count
0    47566B               TEA TIME PARTY BUNTING        786
1    47566B          correct previous adjustment          1
2    47566B  incorrectly credited C550456 see 47          2
3    47566B                              missing          1
4    47566B          reverse previous adjustment          1
5    47566B  stock credited from royal yacht inc          1
6    47566B                                  NaN          3
7    47566b               TEA TIME PARTY BUNTING        113


## Takeaways and next steps
The April diagnostic localizes an accounting decline, but does not establish changes in customer demand. Invoice C550456 records GBP -22,998.40; invoice 550461 for the same customer appears 12 minutes later for GBP 21,535.90. Their combined recorded amount is GBP -1,462.50. Four exact stock codes match; the fifth differs in case and description. Later descriptions reference an incorrect credit, supporting a correction hypothesis without proving a formal invoice link. Keep both invoices; do not relabel all cancellations as business losses.

Next: finish stock-code/correction review, consolidate preprocessing, test final metrics against duplicate-handling alternatives, define repeat-purchase windows, reproduce core results in SQL, and build a dashboard and business report. No experiment has been run; an A/B test design is a future deliverable.


In [88]:
assert len(combined) == 1044848
assert len(product_transactions) == 1033062
assert round(product_transactions["line_amount"].sum(), 2) == 18981261.52
assert len(monthly_transactions) == 1007683
assert len(monthly_net) == 24
print("Milestone reconciliation checks passed.")

Milestone reconciliation checks passed.
